# 01 — Data Understanding

Explore the raw e-commerce fraud data before cleaning or modeling.

Objectives:
- Inspect dataset dimensions, columns, and data types.
- Check missing values and duplicate rows.
- Review the target distribution and descriptive statistics.
- Run and debug the small IP-geolocation experiment in this notebook.

Inputs: `data/raw/Fraud_Data.csv` and `data/raw/IpAddress_to_Country.csv`. The target is `class`: `1` = fraud, `0` = legitimate.

Execution requires Python 3, pandas, NumPy, and IPython; nbclient and ipykernel are used for automated execution.


## Working agreement for data understanding

Until this phase is finished, keep **all analysis, code, experiments, and debugging
in this notebook**. Explain each change in simple English before its code cell.
Do not create Python modules in `src/`, separate test scripts, or another notebook
for this phase. Move code to reusable project files only after the team finishes
data understanding and explicitly chooses that next step.

Keep the original CSVs unchanged. Saved API responses remain evidence under
`data/raw/`. The IP experiment below includes its own functions and debugging
checks; it does not import project code from `src/`.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

# Find this repository, rather than a CSV in a parent project folder.
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "environment.yml").is_file() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Launch this notebook from within the team repository.")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
required_files = ["Fraud_Data.csv", "IpAddress_to_Country.csv"]
missing_files = [name for name in required_files if not (RAW_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing raw files in {RAW_DIR}: {', '.join(missing_files)}. "
        "Download the Kaggle CSVs into data/raw/ before running this notebook."
    )

fraud = pd.read_csv(RAW_DIR / "Fraud_Data.csv")
ip_countries = pd.read_csv(RAW_DIR / "IpAddress_to_Country.csv")


## Dataset overview


In [2]:
for name, frame in {"Purchases": fraud, "IP country lookup": ip_countries}.items():
    print(f"{name}: {frame.shape[0]:,} rows × {frame.shape[1]} columns")
    display(frame.head())
    frame.info()


Purchases: 151,112 rows × 11 columns
<class 'pandas.DataFrame'>
RangeIndex: 151112 entries, 0 to 151111
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   user_id         151112 non-null  int64  
 1   signup_time     151112 non-null  str    
 2   purchase_time   151112 non-null  str    
 3   purchase_value  151112 non-null  int64  
 4   device_id       151112 non-null  str    
 5   source          151112 non-null  str    
 6   browser         151112 non-null  str    
 7   sex             151112 non-null  str    
 8   age             151112 non-null  int64  
 9   ip_address      151112 non-null  float64
 10  class           151112 non-null  int64  
dtypes: float64(1), int64(4), str(6)
memory usage: 12.7 MB
IP country lookup: 138,846 rows × 3 columns
<class 'pandas.DataFrame'>
RangeIndex: 138846 entries, 0 to 138845
Data columns (total 3 columns):
 #   Column                  Non-Null Count   Dtype  
---  ----

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class
0,22058,2015-02-24 22:55:49,2015-04-18 02:47:11,34,QVPSPJUOCKZAR,SEO,Chrome,M,39,7.327584e+08,0
1,333320,2015-06-07 20:39:50,2015-06-08 01:38:54,16,EOGFQPIZPYXFZ,Ads,Chrome,F,53,3.503114e+08,0
2,1359,2015-01-01 18:52:44,2015-01-01 18:52:45,15,YSSKYOSJHPPLJ,SEO,Opera,M,53,2.621474e+09,1
3,150084,2015-04-28 21:13:25,2015-05-04 13:54:50,44,ATGTXKYKUDUQN,SEO,Safari,M,41,3.840542e+09,0
4,221365,2015-07-21 07:09:52,2015-09-09 18:40:53,39,NAUITBZFJKHWW,Ads,Safari,M,45,4.155831e+08,0


,lower_bound_ip_address,upper_bound_ip_address,country
0,16777216.0,16777471,Australia
1,16777472.0,16777727,China
2,16777728.0,16778239,China
3,16778240.0,16779263,Australia
4,16779264.0,16781311,China


## Initial quality checks

These checks describe the raw files; they do not modify the data.


In [3]:
for name, frame in {"Purchases": fraud, "IP country lookup": ip_countries}.items():
    print(name)
    display(pd.DataFrame({
        "dtype": frame.dtypes.astype(str),
        "missing_count": frame.isna().sum(),
        "missing_pct": frame.isna().mean().mul(100),
    }))
    print(f"Duplicate rows: {frame.duplicated().sum():,}")


Purchases
Duplicate rows: 0
IP country lookup
Duplicate rows: 0


,dtype,missing_count,missing_pct
user_id,int64,0,0.0
signup_time,str,0,0.0
purchase_time,str,0,0.0
purchase_value,int64,0,0.0
device_id,str,0,0.0
source,str,0,0.0
browser,str,0,0.0
sex,str,0,0.0
age,int64,0,0.0
ip_address,float64,0,0.0


,dtype,missing_count,missing_pct
lower_bound_ip_address,float64,0,0.0
upper_bound_ip_address,int64,0,0.0
country,str,0,0.0


## Purchase statistics and target distribution


In [4]:
display(fraud.describe(include="all").T)

target_counts = fraud["class"].value_counts(dropna=False).sort_index()
display(pd.DataFrame({
    "count": target_counts,
    "percentage": target_counts.div(len(fraud)).mul(100),
}))


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
user_id,151112.0,NaN,NaN,NaN,200171.04097,115369.285024,2.0,100642.5,199958.0,300054.0,400000.0
signup_time,151112,151112,2015-02-24 22:55:49,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchase_time,151112,150679,2015-07-17 23:22:55,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
purchase_value,151112.0,NaN,NaN,NaN,36.935372,18.322762,9.0,22.0,35.0,49.0,154.0
device_id,151112,137956,ITUMJCKWEYNDD,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
source,151112,3,SEO,60615,NaN,NaN,NaN,NaN,NaN,NaN,NaN
browser,151112,5,Chrome,61432,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sex,151112,2,M,88293,NaN,NaN,NaN,NaN,NaN,NaN,NaN
age,151112.0,NaN,NaN,NaN,33.140704,8.617733,18.0,27.0,33.0,39.0,76.0
ip_address,151112.0,NaN,NaN,NaN,2152145330.958826,1248497030.100426,52093.496895,1085933882.53156,2154770162.40662,3243257679.722522,4294850499.67884


,count,percentage
class,,
0,136961,90.635423
1,14151,9.364577


## Timestamp validity and purchase ordering

Parse the documented timestamp format into separate variables, leaving the raw columns intact. The source timestamps contain no timezone information.


In [5]:
timestamps = fraud[["signup_time", "purchase_time"]].apply(
    lambda column: pd.to_datetime(column, format="%Y-%m-%d %H:%M:%S", errors="coerce")
)
display(pd.DataFrame({
    "invalid_or_missing": timestamps.isna().sum(),
    "earliest": timestamps.min(),
    "latest": timestamps.max(),
}))
lag_seconds = (timestamps["purchase_time"] - timestamps["signup_time"]).dt.total_seconds()
display(pd.Series({
    "purchase_before_signup": int(lag_seconds.lt(0).sum()),
    "purchase_at_signup": int(lag_seconds.eq(0).sum()),
    "purchase_one_second_after_signup": int(lag_seconds.eq(1).sum()),
    "minimum_lag_seconds": lag_seconds.min(),
    "median_lag_days": lag_seconds.median() / 86400,
    "maximum_lag_days": lag_seconds.max() / 86400,
}, name="timestamp_checks"))
display(pd.crosstab(lag_seconds.eq(1).rename("one_second_lag"), fraud["class"]))


,invalid_or_missing,earliest,latest
signup_time,0,2015-01-01 00:00:42,2015-08-18 04:40:29
purchase_time,0,2015-01-01 00:00:44,2015-12-16 02:56:05


purchase_before_signup                 0.000000
purchase_at_signup                     0.000000
purchase_one_second_after_signup    7600.000000
minimum_lag_seconds                    1.000000
median_lag_days                       57.017888
maximum_lag_days                     119.999676
Name: timestamp_checks, dtype: float64

class,0,1
one_second_lag,,
False,136961,6551
True,0,7600


## Numeric ranges, categories, and repeated identifiers

Repeated devices or IPs may be meaningful fraud signals; they are distinct from duplicate transaction rows. Extreme numeric values are flagged for review rather than automatically removed.


In [6]:
import numpy as np

numeric = fraud.select_dtypes(include="number")
display(pd.DataFrame({
    "minimum": numeric.min(),
    "maximum": numeric.max(),
    "non_finite": (~np.isfinite(numeric)).sum(),
    "fractional_values": numeric.mod(1).ne(0).sum(),
}))
display(pd.Series({
    "non_positive_purchase_value": int(fraud["purchase_value"].le(0).sum()),
    "non_positive_age": int(fraud["age"].le(0).sum()),
    "invalid_target": int((~fraud["class"].isin([0, 1])).sum()),
    "ip_outside_ipv4_numeric_bounds": int((~fraud["ip_address"].between(0, 2**32 - 1)).sum()),
}, name="numeric_checks"))
for column in ["source", "browser", "sex"]:
    print(column)
    display(fraud[column].value_counts(dropna=False))

category_checks = []
for name, frame, columns in [
    ("Purchases", fraud, ["source", "browser", "sex", "device_id"]),
    ("IP lookup", ip_countries, ["country"]),
]:
    for column in columns:
        values = frame[column].astype("string")
        stripped = values.str.strip()
        category_checks.append({
            "dataset": name, "column": column,
            "blank_strings": int(stripped.eq("").sum()),
            "surrounding_whitespace": int(values.ne(stripped).sum()),
            "case_or_whitespace_variants": int(values.nunique() - stripped.str.casefold().nunique()),
        })
display(pd.DataFrame(category_checks))

display(pd.DataFrame({
    "unique_values": fraud[["user_id", "device_id", "ip_address"]].nunique(),
    "repeats_beyond_first": [fraud[c].duplicated().sum() for c in ["user_id", "device_id", "ip_address"]],
    "maximum_rows_per_value": [fraud[c].value_counts().max() for c in ["user_id", "device_id", "ip_address"]],
}))
for column in ["purchase_value", "age"]:
    q1, q3 = fraud[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    outside = ~fraud[column].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    print(f"{column}: {int(outside.sum()):,} rows outside 1.5 × IQR fences "
          f"[{q1 - 1.5 * iqr:g}, {q3 + 1.5 * iqr:g}]")


source
browser
sex
purchase_value: 1,067 rows outside 1.5 × IQR fences [-18.5, 89.5]
age: 776 rows outside 1.5 × IQR fences [9, 57]


,minimum,maximum,non_finite,fractional_values
user_id,2.000000,4.000000e+05,0,0
purchase_value,9.000000,1.540000e+02,0,0
age,18.000000,7.600000e+01,0,0
ip_address,52093.496895,4.294850e+09,0,151111
class,0.000000,1.000000e+00,0,0


non_positive_purchase_value       0
non_positive_age                  0
invalid_target                    0
ip_outside_ipv4_numeric_bounds    0
Name: numeric_checks, dtype: int64

source
SEO       60615
Ads       59881
Direct    30616
Name: count, dtype: int64

browser
Chrome     61432
IE         36727
Safari     24667
FireFox    24610
Opera       3676
Name: count, dtype: int64

sex
M    88293
F    62819
Name: count, dtype: int64

,dataset,column,blank_strings,surrounding_whitespace,case_or_whitespace_variants
0,Purchases,source,0,0,0
1,Purchases,browser,0,0,0
2,Purchases,sex,0,0,0
3,Purchases,device_id,0,0,0
4,IP lookup,country,0,0,0


,unique_values,repeats_beyond_first,maximum_rows_per_value
user_id,151112,0,1
device_id,137956,13156,20
ip_address,143512,7600,20


## IP lookup validity and purchase coverage

Treat lookup bounds as inclusive. First audit their order, integer validity, and overlap. Purchase IPs are stored as fractional numbers: report coverage using their original values and compare with flooring as a diagnostic only. Flooring is a candidate conversion that still needs source confirmation.


In [7]:
bounds = ip_countries[["lower_bound_ip_address", "upper_bound_ip_address"]]
lower = bounds["lower_bound_ip_address"]
upper = bounds["upper_bound_ip_address"]
lookup = ip_countries.sort_values("lower_bound_ip_address").reset_index(drop=True)
previous_max_upper = lookup["upper_bound_ip_address"].cummax().shift(1)
overlaps = lookup["lower_bound_ip_address"].le(previous_max_upper)
gaps = lookup["lower_bound_ip_address"].gt(previous_max_upper + 1)
invalid_ranges = (~np.isfinite(bounds)).any(axis=1) | (~bounds.ge(0) | ~bounds.le(2**32 - 1)).any(axis=1) | bounds.mod(1).ne(0).any(axis=1) | lower.gt(upper)
display(pd.Series({
    "lookup_rows": len(lookup),
    "countries": lookup["country"].nunique(),
    "invalid_ranges": int(invalid_ranges.sum()),
    "duplicate_bound_pairs": int(bounds.duplicated().sum()),
    "original_lower_bounds_sorted": lower.is_monotonic_increasing,
    "overlapping_ranges": int(overlaps.sum()),
    "internal_gaps": int(gaps.sum()),
    "smallest_bound": lower.min(),
    "largest_bound": upper.max(),
}, name="lookup_checks"))
assert not invalid_ranges.any(), "Resolve invalid IP ranges before mapping."
assert not overlaps.any(), "Resolve ambiguous IP overlaps before mapping."

def match_country(ip_values):
    # Locate the latest range start at or below each address, then check its end.
    values = np.asarray(ip_values, dtype=float)
    starts = lookup["lower_bound_ip_address"].to_numpy()
    ends = lookup["upper_bound_ip_address"].to_numpy()
    positions = np.searchsorted(starts, values, side="right") - 1
    safe_positions = positions.clip(0)
    matched = (np.isfinite(values) & (positions >= 0)
               & (values <= ends[safe_positions])
               & (values >= 0) & (values <= 2**32 - 1))
    countries = pd.Series(pd.NA, index=fraud.index, dtype="string", name="country")
    countries.loc[matched] = lookup["country"].to_numpy()[positions[matched]]
    return countries

raw_country = match_country(fraud["ip_address"])
floored_country = match_country(np.floor(fraud["ip_address"]))
display(pd.DataFrame({
    "matched_purchases": [raw_country.notna().sum(), floored_country.notna().sum()],
    "unmatched_purchases": [raw_country.isna().sum(), floored_country.isna().sum()],
    "coverage_pct": [raw_country.notna().mean() * 100, floored_country.notna().mean() * 100],
}, index=["Original numeric IP", "Floored IP (diagnostic)"]))
print("Assignments changed by flooring:", int(raw_country.fillna("<unmatched>").ne(floored_country.fillna("<unmatched>")).sum()))
unmatched = raw_country.isna()
display(pd.Series({
    "unmatched_below_lookup_minimum": int((unmatched & fraud["ip_address"].lt(lower.min())).sum()),
    "unmatched_above_lookup_maximum": int((unmatched & fraud["ip_address"].gt(upper.max())).sum()),
    "unmatched_inside_lookup_span": int((unmatched & fraud["ip_address"].between(lower.min(), upper.max())).sum()),
}, name="unmatched_location"))
display(pd.crosstab(raw_country.notna().rename("country_matched"), fraud["class"]))


Assignments changed by flooring: 0


lookup_rows                         138846
countries                              235
invalid_ranges                           0
duplicate_bound_pairs                    0
original_lower_bounds_sorted          True
overlapping_ranges                       0
internal_gaps                         3370
smallest_bound                  16777216.0
largest_bound                   3758096383
Name: lookup_checks, dtype: object

,matched_purchases,unmatched_purchases,coverage_pct
Original numeric IP,129146,21966,85.463762
Floored IP (diagnostic),129146,21966,85.463762


unmatched_below_lookup_minimum      634
unmatched_above_lookup_maximum    19383
unmatched_inside_lookup_span       1949
Name: unmatched_location, dtype: int64

class,0,1
country_matched,,
False,20083,1883
True,116878,12268


## Findings and next steps

Findings from the two local CSV files. Supporting checks and outputs appear above; raw files are unchanged.

- **Dataset size and observed class balance:** Purchases contain **151,112 rows × 11 columns**. There are **136,961 legitimate purchases (90.64%)** and **14,151 fraudulent purchases (9.36%)**. The IP lookup contains **138,846 rows × 3 columns**, covering **235 country labels**. The target is imbalanced; a majority-only prediction would miss every fraud case.

- **Missing values and duplicate records:** Both datasets have **zero missing values and zero duplicate rows**. Audited text columns contain no blank strings or surrounding whitespace. All **151,112 user IDs are unique**. Devices and IPs repeat: **137,956 unique devices** (13,156 repeats beyond the first occurrence) and **143,512 unique IP values** (7,600 repeats beyond the first); both have a maximum of 20 purchases per value. Preserve these distinct transactions and investigate shared identifiers as potential signals.

- **Timestamp validity and signup-to-purchase ordering:** All timestamps parse with `%Y-%m-%d %H:%M:%S`. Signups span **2015-01-01 00:00:42 to 2015-08-18 04:40:29**; purchases span **2015-01-01 00:00:44 to 2015-12-16 02:56:05**. No purchase precedes or equals signup. The delay ranges from **1 second to just under 120 days**, with a median of **57.02 days**. **7,600 purchases occur exactly one second after signup, all labeled fraud**; they account for **53.71% of fraud cases**. Investigate this unusually strong pattern and whether it reflects collection or generation artifacts. Timezones are unspecified.

- **Numeric ranges, category consistency, and unusual values:** Purchase value ranges from **9 to 154**, age from **18 to 76**, and user ID from **2 to 400,000**. Numeric fields are finite; purchase values and ages are positive; target values are only `0` and `1`. Source values are `Ads`, `Direct`, and `SEO`; browsers are `Chrome`, `FireFox`, `IE`, `Opera`, and `Safari`; sex values are `F` and `M`. Audited categorical fields show no variants caused only by case or whitespace. The 1.5 × IQR rule flags **1,067 purchase values above 89.5** and **776 ages above 57**; these are review flags, not established errors. Purchase IPs span approximately **52,093.4969 to 4,294,850,499.6788**, within numeric IPv4 bounds, but **151,111 of 151,112 values are fractional**. Their encoding needs confirmation before conversion to standard IP strings or API use. Purchase currency and units also need confirmation.

- **IP lookup range validity, overlap, and purchase coverage:** Bounds are integer-valued, sorted by lower bound, within IPv4 numeric bounds, and correctly ordered (`lower <= upper`), with **zero duplicate bound pairs and zero overlapping ranges**. The lookup spans **16,777,216 to 3,758,096,383** and has **3,370 internal gaps**. Inclusive range matching covers **129,146 purchases (85.46%)** and leaves **21,966 (14.54%) unmatched**: **634 below** the lookup minimum, **19,383 above** its maximum, and **1,949 within gaps in the lookup span**. Flooring purchase IP values changes **zero country assignments** in this dataset, but does not establish the intended encoding. Missing country matches therefore represent lookup coverage limitations rather than missing source IPs.

- **Cleaning decisions and questions for further investigation:** Retain the original CSVs and all transaction rows; no missing-value imputation or row deduplication is indicated by these checks. Parse timestamps into separate datetime columns, document the unknown timezone, and derive signup-to-purchase delay. Confirm IP encoding before adopting an integer conversion; preserve raw IPs and retain unmatched purchases with an explicit unknown country or match-status flag. Keep the observed category spelling, including `FireFox`, unless a documented mapping is introduced. Keep numeric outliers pending source review. Treat `user_id` as an identifier, and construct device/IP history features using only earlier purchases. For modeling, use class-aware metrics and assess a chronological split with shared-device/IP overlap checks. Investigate label provenance, the one-second fraud pattern, currency/units, and whether historical IP-country coverage is appropriate before deciding on further enrichment or exclusions.


# Small IP geolocation experiment — 10 public addresses

Continue the initial profiling above. Can we convert 10 dataset IP values, look
up their countries, save responses and failures, and compare the API countries
with the static lookup?

The conversion, collection, saved-response review, and debugging checks are all
here. No project module or separate test file is needed. The original profiling
cells are retained.

**Run All reviews the saved sample without making an API request.** The optional
live-collection cell at the end is switched off. If deliberately enabled later,
it sends only one batch of 10 addresses. Full collection and CPI scraping remain
deferred.

## Step 1 — Find the data and the saved run

We use the same two source CSVs as the profiling above. We explicitly choose the saved
successful run, so a future failed attempt cannot silently replace this evidence.
The `data/` folder is ignored by Git: running this notebook on another computer
requires the local CSVs and saved run folder.

We also record file hashes now. A hash is a fingerprint of a file's bytes; we
will compare the fingerprints again at the end.

In [8]:
from bisect import bisect_right
import csv
from datetime import datetime, timezone
from decimal import Decimal, InvalidOperation, ROUND_FLOOR
import hashlib
import ipaddress
from ipaddress import IPv4Address
import json
import platform
import time
from urllib.error import HTTPError, URLError
from urllib.request import HTTPRedirectHandler, Request, build_opener

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
ROOT = PROJECT_ROOT
RAW = RAW_DIR
FRAUD_PATH = RAW_DIR / "Fraud_Data.csv"
STATIC_PATH = RAW_DIR / "IpAddress_to_Country.csv"
SAMPLE_SIZE = 10
URL = ("http://ip-api.com/batch?fields="
       "status,message,query,country,countryCode,proxy,hosting&lang=en")
DOC_URL = "https://ip-api.com/docs/api:batch"
RUN_DIR = RAW_DIR / "ip_api_sample/20261031T194557_553595Z"
required_sample_files = [
    RUN_DIR / name for name in ["sample.csv", "request.json", "response.body",
                               "lookup.json", "provenance.json", "summary.json"]
]
missing_sample_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_sample_files
                        if not path.is_file()]
if missing_sample_files:
    raise FileNotFoundError("Local sample files are missing: " + ", ".join(missing_sample_files))

def file_fingerprints(paths):
    return {str(path.relative_to(PROJECT_ROOT)): hashlib.sha256(path.read_bytes()).hexdigest()
            for path in paths}

# This notebook is the analysis being updated, so it is not a protected input.
protected_files = [FRAUD_PATH, STATIC_PATH, PROJECT_ROOT / "notebooks/01_understanding.ipynb"]
evidence_files_before = sorted(path for path in (RAW_DIR / "ip_api_sample").rglob("*")
                              if path.is_file())
protected_files += evidence_files_before
hashes_before = file_fingerprints(protected_files)
print("Saved run:", RUN_DIR.relative_to(PROJECT_ROOT))
print("Protected input/evidence files:", len(hashes_before))

Saved run: data/raw/ip_api_sample/20261031T194557_553595Z
Protected input/evidence files: 24


## Step 2 — Explain the fractional-IP assumption

IPv4 uses a whole number between `0` and `4294967295`. Our CSV contains decimal
values, so we must choose and document a conversion before sending an IP.

**For this sample, assume that the whole-number part is the intended IPv4
number.** We use flooring: `732758368.79972` becomes `732758368`. We do not round
to the nearest integer, because that could select the next address.

Read the values as text, then use `Decimal`, so a floating-point conversion
cannot round the source value before we decide what to do. Check the original
value is finite and within IPv4 bounds before flooring. These checks are built
into `floor_ipv4`.

The original encoding is **unconfirmed**. The profiling above found that flooring changes
zero static country assignments in the full dataset; that supports this small
experiment but does not prove the assumption.

In [9]:
def floor_ipv4(raw_value):
    """Experimental assumption: discard the fraction, without float rounding."""
    try:
        value = Decimal(raw_value)
    except InvalidOperation as exc:
        raise ValueError("IP must be a decimal number") from exc
    if not value.is_finite() or not 0 <= value <= 2**32 - 1:
        raise ValueError("IP must be finite and within numeric IPv4 bounds")
    return int(value.to_integral_value(rounding=ROUND_FLOOR))


def is_public(address):
    # is_global alone also accepts multicast addresses on some Python versions.
    return (address.is_global and not address.is_multicast
            and not address.is_reserved and not address.is_loopback
            and not address.is_link_local and not address.is_unspecified)

In [10]:
# Only load the IP column, preserving the original decimal text.
ip_values = pd.read_csv(FRAUD_PATH, usecols=["ip_address"], dtype="string")["ip_address"]
raw_example = ip_values.iloc[0]
decimal_example = Decimal(raw_example)
integer_example = floor_ipv4(raw_example)
assert integer_example == int(decimal_example.to_integral_value(rounding=ROUND_FLOOR))

display(pd.DataFrame([{
    "original_csv_text": raw_example,
    "floored_integer": integer_example,
    "dotted_ipv4": str(IPv4Address(integer_example)),
}]))

fractional_count = sum(Decimal(value) % 1 != 0 for value in ip_values)
print(f"Fractional values: {fractional_count:,} of {len(ip_values):,}")
print("The original CSV values are preserved; conversion is kept separately.")

Fractional values: 151,111 of 151,112
The original CSV values are preserved; conversion is kept separately.


,original_csv_text,floored_integer,dotted_ipv4
0,732758368.79972,732758368,43.173.1.96


## Step 3 — Select 10 unique public addresses

Read transactions in CSV order. Convert each IP, exclude addresses that are not
public, and keep the first 10 distinct converted addresses. Deduplicate **after**
flooring: two different fractions can produce the same IP.

The public filter excludes private, loopback, link-local, shared, reserved, and
multicast addresses. It uses Python's `ipaddress` definitions. A public address
does not guarantee an active host. These definitions can change between Python
versions, so the saved run includes its Python version and exact sample.

The `select_sample` function below follows these rules. It also finds each
static country with inclusive range matching: `lower <= IP <= upper`. It checks
that the static ranges are valid and do not overlap first.

We select without using the fraud label, API result, or static match status.
These first 10 public addresses are a **convenience sample**, not a representative
sample of countries.

In [11]:
def load_static(path):
    with path.open(newline="", encoding="utf-8") as stream:
        ranges = sorted((Decimal(row["lower_bound_ip_address"]),
                         Decimal(row["upper_bound_ip_address"]), row["country"])
                        for row in csv.DictReader(stream))
    previous_end = Decimal(-1)
    for lower, upper, _ in ranges:
        if not (lower.is_finite() and upper.is_finite()
                and 0 <= lower <= upper <= 2**32 - 1
                and lower == int(lower) and upper == int(upper)
                and lower > previous_end):
            raise ValueError("Static lookup needs valid, non-overlapping integer ranges")
        previous_end = upper
    return [row[0] for row in ranges], ranges


def static_country(value, lookup):
    starts, ranges = lookup
    position = bisect_right(starts, value) - 1
    if position >= 0 and value <= ranges[position][1]:
        return ranges[position][2]
    return ""


def select_sample(fraud_path, lookup):
    selected, seen = [], set()
    with fraud_path.open(newline="", encoding="utf-8") as stream:
        for line_number, row in enumerate(csv.DictReader(stream), start=2):
            raw_value = row["ip_address"]
            try:
                number = floor_ipv4(raw_value)
            except ValueError:
                continue
            address = ipaddress.IPv4Address(number)
            if number in seen or not is_public(address):
                continue
            seen.add(number)
            selected.append({
                "source_csv_line": line_number, "raw_ip": raw_value,
                "ip_integer": number, "ip_address": str(address),
                "static_country_raw": static_country(Decimal(raw_value), lookup),
                "static_country": static_country(Decimal(number), lookup),
            })
            if len(selected) == SAMPLE_SIZE:
                return selected
    raise ValueError("Dataset contains fewer than 10 distinct eligible public IPv4s")

In [12]:
static_lookup = load_static(STATIC_PATH)
selected = select_sample(FRAUD_PATH, static_lookup)
sample = pd.DataFrame(selected)

assert len(sample) == 10
assert sample["ip_address"].nunique() == 10
assert sample["ip_address"].map(lambda value: is_public(IPv4Address(value))).all()

# Check that today's reproduction matches the sample actually sent to the API.
saved_sample = pd.read_csv(RUN_DIR / "sample.csv", dtype={"raw_ip": "string"},
                           keep_default_na=False)
pd.testing.assert_frame_equal(sample, saved_sample, check_dtype=False)

display(sample[["source_csv_line", "raw_ip", "ip_integer", "ip_address", "static_country"]])
changed = sample["static_country_raw"].ne(sample["static_country"]).sum()
print("Static country assignments changed by flooring in this sample:", changed)

Static country assignments changed by flooring in this sample: 0


,source_csv_line,raw_ip,ip_integer,ip_address,static_country
0,2,732758368.79972,732758368,43.173.1.96,Japan
1,3,350311387.865908,350311387,20.225.83.219,United States
2,4,2621473820.11095,2621473820,156.64.132.28,United States
3,6,415583117.452712,415583117,24.197.75.141,United States
4,7,2809315199.92675,2809315199,167.114.191.127,Canada
5,9,1692458727.64945,1692458727,100.224.222.231,United States
6,10,3719094257.18731,3719094257,221.172.223.241,China
7,11,341674739.579911,341674739,20.93.138.243,United States
8,12,1819008577.79416,1819008577,108.107.222.65,United States
9,15,3178510014.63508,3178510014,189.116.54.190,Brazil


## Step 4 — Inspect the request and the original response

The [official batch documentation](https://ip-api.com/docs/api:batch), checked
on 2026-10-31, allows a JSON POST with up to **100 addresses** and **15 batch
requests per minute** from one client IP. The free service uses HTTP, needs no
API key, and is for non-commercial use.

We sent all 10 addresses in **one request**, asking for status, failure message,
query address, country, country code, and optional proxy/hosting flags. We sent
no other transaction information. The collection function below makes no automatic retries or
redirects and saves evidence after the request.

`X-Rl` tells us how many batch requests remain in the current window; `X-Ttl`
tells us how many seconds remain before reset. If `X-Rl` is zero, no more requests
may be sent until reset. The collection function keeps a conservative cooldown between runs.

Below we **read the saved request**. This cell does not send it.

The [service terms](https://ip-api.com/docs/legal) restrict the free API to
non-commercial use; this is a coursework experiment. HTTP 429 means throttling,
and repeated excess requests can cause a one-hour ban. The cooldown extends
using `X-Ttl` if the quota is exhausted. Run one copy at a time: the cooldown is
not a concurrent-process lock. Other programs sharing the client IP can consume
quota too.

### Define collection here; do not send a request yet

These functions save the exact request, source hashes, response bytes, HTTP
headers, UTC start/end, duration, and one result row per IP. Each attempt gets a
new folder, so failures cannot overwrite earlier results. If no response arrives,
no response body is saved; transport failures are recorded in the metadata.
HTTP errors keep their original bodies and headers.

This cell only defines functions. The optional cell at the end controls whether
`collect` is called. The default path reads the evidence already saved.

In [13]:
def utc_now():
    return datetime.now(timezone.utc).isoformat()


def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                    encoding="utf-8")


def save_csv(path, rows):
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


class NoRedirect(HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        return None


def fetch_batch(payload):
    request = Request(URL, data=payload, method="POST", headers={
        "Content-Type": "application/json", "User-Agent": "DEPI-IP-Sample/1.0",
    })
    try:
        response = build_opener(NoRedirect()).open(request, timeout=30)
    except HTTPError as exc:
        response = exc  # HTTP failures still have original bodies and headers.
    with response:
        return response.code, dict(response.headers.items()), response.read()


def collect(sample, sources):
    output_root = RAW / "ip_api_sample"
    output_root.mkdir(exist_ok=True)
    gate = output_root / "rate_limit.json"
    if gate.exists():
        wait = json.loads(gate.read_text())["not_before_epoch"] - time.time()
        if wait > 0:
            raise RuntimeError(f"No request sent. Rate-limit cooldown: wait {wait:.0f} seconds")
    run = output_root / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    run.mkdir()
    save_csv(run / "sample.csv", sample)
    payload = json.dumps([row["ip_address"] for row in sample]).encode("utf-8")
    (run / "request.json").write_bytes(payload)
    save_json(run / "provenance.json", {
        "source_sha256": sources, "python_version": platform.python_version(),
        "selection": "First 10 unique public floored IPv4s in CSV order; no target use",
        "conversion_assumption": "Floor exact decimal text; original encoding unconfirmed",
        "documentation_url": DOC_URL, "documentation_checked_on": "2026-10-31",
    })
    # One request per run, no retries. Persist a conservative 65-second cooldown
    # before sending, so even an interrupted run leaves a rate-limit record.
    save_json(gate, {"not_before_epoch": time.time() + 65})
    metadata = {"url": URL, "method": "POST", "started_at_utc": utc_now(),
                "http_status": None, "response_headers": {}, "error": ""}
    save_json(run / "lookup.json", metadata)
    start = time.monotonic()
    body = None
    try:
        code, headers, body = fetch_batch(payload)
        metadata.update(http_status=code, response_headers=headers)
        (run / "response.body").write_bytes(body)
    except (URLError, OSError, TimeoutError) as exc:
        metadata["error"] = str(exc)
    metadata.update(finished_at_utc=utc_now(),
                    elapsed_seconds=round(time.monotonic() - start, 6))
    headers = {key.lower(): value for key, value in metadata["response_headers"].items()}
    cooldown = 65
    if headers.get("x-rl") == "0" or metadata["http_status"] == 429:
        try:
            cooldown = max(cooldown, int(headers.get("x-ttl", "60")) + 1)
        except ValueError:
            pass
    save_json(gate, {"not_before_epoch": time.time() + cooldown,
                     "x_rl": headers.get("x-rl"), "x_ttl": headers.get("x-ttl")})
    save_json(run / "lookup.json", metadata)
    results = compare_results(sample, metadata, body)
    save_csv(run / "results.csv", results)
    summary = summarize(results)
    save_json(run / "summary.json", summary)
    print(f"Saved sample evidence: {run.relative_to(ROOT)}")
    print(json.dumps(summary, indent=2))
    return summary["successful_country_results"] == SAMPLE_SIZE

In [14]:
saved_request = json.loads((RUN_DIR / "request.json").read_text(encoding="utf-8"))
lookup_metadata = json.loads((RUN_DIR / "lookup.json").read_text(encoding="utf-8"))
provenance = json.loads((RUN_DIR / "provenance.json").read_text(encoding="utf-8"))
assert saved_request == sample["ip_address"].tolist()

print("Saved method:", lookup_metadata["method"])
print("Saved URL:", lookup_metadata["url"])
print("Saved request body:")
print(json.dumps(saved_request, indent=2))
print("Python used for collection:", provenance["python_version"])

headers = {name.lower(): value for name, value in lookup_metadata["response_headers"].items()}
display(pd.DataFrame([{
    "started_at_utc": lookup_metadata["started_at_utc"],
    "finished_at_utc": lookup_metadata["finished_at_utc"],
    "batch_elapsed_seconds": lookup_metadata["elapsed_seconds"],
    "http_status": lookup_metadata["http_status"],
    "requests_remaining_X_Rl": headers.get("x-rl"),
    "reset_seconds_X_Ttl": headers.get("x-ttl"),
}]).T.rename(columns={0: "saved_value"}))

Saved method: POST
Saved URL: http://ip-api.com/batch?fields=status,message,query,country,countryCode,proxy,hosting&lang=en
Saved request body:
[
  "43.173.1.96",
  "20.225.83.219",
  "156.64.132.28",
  "24.197.75.141",
  "167.114.191.127",
  "100.224.222.231",
  "221.172.223.241",
  "20.93.138.243",
  "108.107.222.65",
  "189.116.54.190"
]
Python used for collection: 3.14.8


,saved_value
started_at_utc,2026-10-31T19:45:57.560269+00:00
finished_at_utc,2026-10-31T19:45:57.800576+00:00
batch_elapsed_seconds,0.240207
http_status,200
requests_remaining_X_Rl,14
reset_seconds_X_Ttl,60


The original response is saved as `response.body`, without rewriting its JSON.
We display those bytes as text and make a separate table for reading. A successful
HTTP request alone is not enough: each address must also have a valid API status,
query address, country, and country code.

The elapsed time belongs to the **whole batch**, not to each individual IP.
Proxy and hosting flags are the API's current observations; they are not proof
that a customer committed fraud.

In [15]:
response_body = (RUN_DIR / "response.body").read_bytes()
assert len(response_body) == int(headers["content-length"])
print(f"Original response ({len(response_body):,} bytes):")
print(response_body.decode("utf-8"))
api_response = json.loads(response_body)
display(pd.DataFrame(api_response)[
    ["query", "status", "country", "countryCode", "proxy", "hosting"]
])

Original response (1,175 bytes):
[{"status":"success","country":"Indonesia","countryCode":"ID","proxy":false,"hosting":true,"query":"43.173.1.96"},{"status":"success","country":"United States","countryCode":"US","proxy":false,"hosting":true,"query":"20.225.83.219"},{"status":"success","country":"United States","countryCode":"US","proxy":false,"hosting":false,"query":"156.64.132.28"},{"status":"success","country":"United States","countryCode":"US","proxy":false,"hosting":false,"query":"24.197.75.141"},{"status":"success","country":"Canada","countryCode":"CA","proxy":false,"hosting":true,"query":"167.114.191.127"},{"status":"success","country":"United States","countryCode":"US","proxy":false,"hosting":false,"query":"100.224.222.231"},{"status":"success","country":"China","countryCode":"CN","proxy":false,"hosting":false,"query":"221.172.223.241"},{"status":"success","country":"Netherlands","countryCode":"NL","proxy":false,"hosting":true,"query":"20.93.138.243"},{"status":"success","countr

,query,status,country,countryCode,proxy,hosting
0,43.173.1.96,success,Indonesia,ID,False,True
1,20.225.83.219,success,United States,US,False,True
2,156.64.132.28,success,United States,US,False,False
3,24.197.75.141,success,United States,US,False,False
4,167.114.191.127,success,Canada,CA,False,True
5,100.224.222.231,success,United States,US,False,False
6,221.172.223.241,success,China,CN,False,False
7,20.93.138.243,success,Netherlands,NL,False,True
8,108.107.222.65,success,United States,US,False,False
9,189.116.54.190,success,Brazil,BR,False,False


## Step 5 — Keep failures as evidence

The first attempt failed at DNS resolution, so no response arrived. A later
attempt received HTTP 307 pointing to an ISP page. That redirect was preserved
and not followed. The third attempt returned countries successfully.

Failures stay in separate run folders. We do not delete failed rows, pretend
they succeeded, or replace a missing agreement rate with zero.

In [16]:
history_rows = []
for run in sorted((RAW_DIR / "ip_api_sample").iterdir()):
    if not run.is_dir():
        continue
    if not (run / "lookup.json").is_file() or not (run / "summary.json").is_file():
        continue
    metadata = json.loads((run / "lookup.json").read_text(encoding="utf-8"))
    summary = json.loads((run / "summary.json").read_text(encoding="utf-8"))
    run_headers = {name.lower(): value for name, value in metadata["response_headers"].items()}
    history_rows.append({
        "run": run.name,
        "http_status": str(metadata["http_status"]) if metadata["http_status"] is not None else "no response",
        "successful": summary["successful_country_results"],
        "failed_or_incomplete": summary["failed_or_incomplete"],
        "agreement": (f"{summary['country_name_match_rate']:.0%}"
                      if summary["country_name_match_rate"] is not None else "unavailable"),
        "error_or_redirect": metadata.get("error") or run_headers.get("location", ""),
    })
with pd.option_context("display.max_colwidth", 100):
    display(pd.DataFrame(history_rows))

,run,http_status,successful,failed_or_incomplete,agreement,error_or_redirect
0,20261031T193938_215014Z,no response,0,10,unavailable,<urlopen error [Errno -3] Temporary failure in name resolution>
1,20261031T194102_822753Z,307,0,10,unavailable,http://megaplusredirection.tedata.net/VDSL-Redirection_100.html
2,20261031T194557_553595Z,200,10,0,80%,


## Step 6 — Compare successful country results

Join API responses to the selected addresses using the returned `query` field,
not their position in the response list. Keep the original IP text and both
country sources beside each other.

The comparison labels mean:

- `match`: country names agree after trimming and ignoring case.
- `mismatch`: both sources have a country, but the names differ.
- `static_unmatched`: the API succeeds but the static table has no matching range.
- `not_comparable`: the API fails or returns incomplete information.

**Agreement = matching countries / successful results with a static match.**
Failed and static-unmatched rows are counted separately. If nothing can be
compared, the agreement rate is unavailable.

The comparison function also detects malformed JSON, non-list responses, missing
or duplicate query results, HTTP errors, API failures, and incomplete country
results. Optional proxy/hosting fields stay blank if missing. Country aliases
are not guessed. Every selected IP keeps a result row.

In [17]:
def compare_results(sample, metadata, body):
    decoded, decode_error = None, ""
    if body is not None:
        try:
            decoded = json.loads(body)
        except (ValueError, UnicodeDecodeError) as exc:
            decode_error = str(exc)
    by_query = {}
    if isinstance(decoded, list):
        for item in decoded:
            if isinstance(item, dict) and isinstance(item.get("query"), str):
                by_query.setdefault(item["query"], []).append(item)
    results = []
    for original in sample:
        items = by_query.get(original["ip_address"], [])
        item = items[0] if len(items) == 1 else {}
        message = metadata.get("error", "")
        if metadata["http_status"] is None:
            status = "transport_error"
        elif metadata["http_status"] != 200:
            status, message = "http_error", f"HTTP {metadata['http_status']}"
        elif decode_error:
            status, message = "invalid_json", decode_error
        elif not isinstance(decoded, list) or len(items) != 1:
            status, message = "invalid_response", "Expected exactly one result for this IP"
        elif item.get("status") == "fail":
            status, message = "api_failure", item.get("message", "")
        elif (item.get("status") != "success"
              or not isinstance(item.get("country"), str) or not item["country"].strip()
              or not isinstance(item.get("countryCode"), str)
              or len(item["countryCode"]) != 2):
            status, message = "incomplete_response", "Missing success/country/countryCode"
        else:
            status = "success"
        comparison = "not_comparable"
        if status == "success":
            if not original["static_country"]:
                comparison = "static_unmatched"
            elif original["static_country"].strip().casefold() == item["country"].strip().casefold():
                comparison = "match"
            else:
                comparison = "mismatch"
        results.append({
            **original, "lookup_started_at_utc": metadata["started_at_utc"],
            "lookup_finished_at_utc": metadata["finished_at_utc"],
            "elapsed_seconds": metadata["elapsed_seconds"],
            "http_status": metadata["http_status"], "lookup_status": status,
            "api_status": item.get("status", ""), "message": message,
            "api_country": item.get("country", ""),
            "api_country_code": item.get("countryCode", ""),
            "proxy": item.get("proxy", ""), "hosting": item.get("hosting", ""),
            "comparison": comparison,
        })
    return results


def summarize(results):
    comparable = sum(row["comparison"] in ("match", "mismatch") for row in results)
    matches = sum(row["comparison"] == "match" for row in results)
    successes = sum(row["lookup_status"] == "success" for row in results)
    return {
        "selected": len(results), "successful_country_results": successes,
        "failed_or_incomplete": len(results) - successes,
        "comparable_successes": comparable, "matches": matches,
        "mismatches": comparable - matches,
        "successful_but_static_unmatched": sum(
            row["comparison"] == "static_unmatched" for row in results),
        "country_name_match_rate": matches / comparable if comparable else None,
    }


def review(run):
    """Recompute the comparison from saved evidence; make no requests or edits."""
    with (run / "sample.csv").open(newline="", encoding="utf-8") as stream:
        selected = list(csv.DictReader(stream))
    metadata = json.loads((run / "lookup.json").read_text(encoding="utf-8"))
    response_path = run / "response.body"
    body = response_path.read_bytes() if response_path.exists() else None
    results = compare_results(selected, metadata, body)
    rows = [("IP", "Static country", "API country", "Lookup", "Comparison")]
    rows.extend((row["ip_address"], row["static_country"] or "(unmatched)",
                 row["api_country"] or "(none)", row["lookup_status"],
                 row["comparison"]) for row in results)
    widths = [max(len(value) for value in column) for column in zip(*rows)]
    print(f"Saved lookup: {metadata['started_at_utc']}")
    print(f"HTTP: {metadata['http_status']}; duration: {metadata['elapsed_seconds']} seconds")
    for row in rows:
        print(" | ".join(value.ljust(width) for value, width in zip(row, widths)))
    summary = summarize(results)
    print(json.dumps(summary, indent=2))
    return summary

In [18]:
# Recalculate from the original response instead of trusting the saved summary.
comparison_rows = compare_results(selected, lookup_metadata, response_body)
comparison = pd.DataFrame(comparison_rows)
summary = summarize(comparison_rows)
saved_summary = json.loads((RUN_DIR / "summary.json").read_text(encoding="utf-8"))
assert summary == saved_summary

display(comparison[["ip_address", "static_country", "api_country", "api_country_code",
                    "lookup_status", "comparison"]])
display(pd.Series(summary, dtype=object, name="sample_results").to_frame())

if summary["country_name_match_rate"] is None:
    print("Country agreement is unavailable: there are no comparable successful results.")
else:
    print(f"Country agreement: {summary['matches']} / {summary['comparable_successes']} "
          f"= {summary['country_name_match_rate']:.0%}")

Country agreement: 8 / 10 = 80%


,ip_address,static_country,api_country,api_country_code,lookup_status,comparison
0,43.173.1.96,Japan,Indonesia,ID,success,mismatch
1,20.225.83.219,United States,United States,US,success,match
2,156.64.132.28,United States,United States,US,success,match
3,24.197.75.141,United States,United States,US,success,match
4,167.114.191.127,Canada,Canada,CA,success,match
5,100.224.222.231,United States,United States,US,success,match
6,221.172.223.241,China,China,CN,success,match
7,20.93.138.243,United States,Netherlands,NL,success,mismatch
8,108.107.222.65,United States,United States,US,success,match
9,189.116.54.190,Brazil,Brazil,BR,success,match


,sample_results
selected,10
successful_country_results,10
failed_or_incomplete,0
comparable_successes,10
matches,8
mismatches,2
successful_but_static_unmatched,0
country_name_match_rate,0.8


## Step 7 — Examine the differences

In this saved run, all 10 country results succeeded. Eight match the static
lookup, so **sample agreement is 80%**. The two differing rows are shown below.

This is agreement between sources, **not API accuracy or fraud-model accuracy**.
The transactions are from 2015; the live response is from 2026-10-31. IP
allocations and geolocation sources can change. The conversion assumption is
also still unconfirmed. These results do not establish which country was correct
at the transaction date.

Both countries are kept. We do not replace the static lookup with the API value
or assume the API corrects the historical data. Even a successful API response
cannot prove that the fractional-IP conversion was correct.

In [19]:
differences = comparison.loc[
    comparison["comparison"].eq("mismatch"),
    ["raw_ip", "ip_address", "static_country", "api_country", "lookup_started_at_utc"],
]
display(differences.reset_index(drop=True))

,raw_ip,ip_address,static_country,api_country,lookup_started_at_utc
0,732758368.79972,43.173.1.96,Japan,Indonesia,2026-10-31T19:45:57.560269+00:00
1,341674739.579911,20.93.138.243,United States,Netherlands,2026-10-31T19:45:57.560269+00:00


## Debug the experiment in this notebook

These seven checks cover exact decimal conversion, public-IP filtering, inclusive
static bounds, deduplication, country comparison, response preservation, and the
cooldown. They also check that saved responses can be reviewed without network
calls or file changes.

Fixtures use temporary CSV/JSON data. Fake request responses keep the checks
offline and away from the real raw-data directory. If a check fails, fix the
function cell above and rerun these notebook cells. There is no separate test
script.

In [20]:
import csv
from contextlib import redirect_stdout
from decimal import Decimal
import io
import ipaddress
import json
from pathlib import Path
import tempfile
import unittest
from unittest.mock import Mock, patch



class SampleTests(unittest.TestCase):
    def test_floor_uses_exact_decimal_and_validates_before_conversion(self):
        self.assertEqual(floor_ipv4("732758368.79972"), 732758368)
        # Converting this text to float first could round it into the next IP.
        self.assertEqual(floor_ipv4("16777215.999999999999"), 16777215)
        self.assertEqual(floor_ipv4("4294967295"), 4294967295)
        for value in ["-0.1", "4294967295.1", "NaN", "Infinity", "bad", ""]:
            with self.subTest(value=value), self.assertRaises(ValueError):
                floor_ipv4(value)

    def test_public_excludes_special_ranges(self):
        for value in ["10.0.0.1", "127.0.0.1", "169.254.1.1", "100.64.0.1",
                      "192.0.2.1", "224.0.0.1", "240.0.0.1", "0.0.0.0"]:
            self.assertFalse(is_public(ipaddress.IPv4Address(value)), value)
        self.assertTrue(is_public(ipaddress.IPv4Address("8.8.8.8")))

    def test_static_inclusive_boundaries_and_gaps(self):
        lookup = ([Decimal(10), Decimal(20)],
                  [(Decimal(10), Decimal(15), "A"), (Decimal(20), Decimal(25), "B")])
        for value, expected in [(9, ""), (10, "A"), (15, "A"), (16, ""), (20, "B"), (26, "")]:
            self.assertEqual(static_country(Decimal(value), lookup), expected)

    def test_selection_deduplicates_after_flooring(self):
        with tempfile.TemporaryDirectory() as directory:
            path = Path(directory) / "fraud.csv"
            values = ["167772161.5", "134744072.1", "134744072.9"]  # private, same IP twice
            values += [str(134744072 + i) + ".25" for i in range(1, 10)]
            with path.open("w", newline="") as stream:
                writer = csv.writer(stream)
                writer.writerow(["ip_address"])
                writer.writerows([[value] for value in values])
            selected = select_sample(path, ([], []))
        self.assertEqual(len(selected), 10)
        self.assertEqual(selected[0]["source_csv_line"], 3)
        self.assertEqual(len({row["ip_address"] for row in selected}), 10)

    def test_comparison_pairs_by_query_and_excludes_failures_from_denominator(self):
        addresses = ["8.8.8.8", "1.1.1.1", "9.9.9.9", "8.8.4.4"]
        selected = [{"ip_address": ip, "static_country": country}
                    for ip, country in zip(addresses, ["United States", "Old name", "", "A"])]
        metadata = {"http_status": 200, "started_at_utc": "start",
                    "finished_at_utc": "end", "elapsed_seconds": 0.5}
        body = json.dumps([
            {"query": addresses[3], "status": "fail", "message": "reserved range"},
            {"query": addresses[2], "status": "success", "country": "Canada", "countryCode": "CA"},
            {"query": addresses[1], "status": "success", "country": "New name", "countryCode": "NZ"},
            {"query": addresses[0], "status": "success", "country": "United States", "countryCode": "US"},
        ]).encode()
        results = compare_results(selected, metadata, body)
        self.assertEqual([row["comparison"] for row in results],
                         ["match", "mismatch", "static_unmatched", "not_comparable"])
        summary = summarize(results)
        self.assertEqual(summary["successful_country_results"], 3)
        self.assertEqual(summary["comparable_successes"], 2)
        self.assertEqual(summary["country_name_match_rate"], 0.5)
        for body, expected in [(b"blocked", "invalid_json"), (b"{}", "invalid_response"),
                               (b"[]", "invalid_response")]:
            self.assertEqual(compare_results(selected, metadata, body)[0]["lookup_status"], expected)
        metadata["http_status"] = 429
        self.assertEqual(compare_results(selected, metadata, b"blocked")[0]["lookup_status"], "http_error")
        metadata["http_status"] = None
        self.assertIsNone(summarize(compare_results(selected, metadata, None))["country_name_match_rate"])

    def test_collection_preserves_body_and_stops_during_rate_limit_cooldown(self):
        selected = [{"ip_address": "8.8.8.8", "static_country": "United States"}]
        body = b'[ {"query":"8.8.8.8","status":"success","country":"United States","countryCode":"US"} ]\n'
        with tempfile.TemporaryDirectory() as directory:
            root = Path(directory)
            raw = root / "raw"
            raw.mkdir()
            fetch = Mock(return_value=(200, {"X-Rl": "0", "X-Ttl": "120"}, body))
            with patch.dict(globals(), {"ROOT": root, "RAW": raw, "fetch_batch": fetch}):
                with redirect_stdout(io.StringIO()):
                    collect(selected, {})
                run = next(path for path in (raw / "ip_api_sample").iterdir() if path.is_dir())
                self.assertEqual((run / "response.body").read_bytes(), body)
                metadata = json.loads((run / "lookup.json").read_text())
                self.assertEqual(metadata["response_headers"]["X-Rl"], "0")
                with self.assertRaisesRegex(RuntimeError, "cooldown"):
                    collect(selected, {})
                self.assertEqual(fetch.call_count, 1)

    def test_review_recomputes_saved_response_without_network_or_writes(self):
        with tempfile.TemporaryDirectory() as directory:
            run = Path(directory)
            save_csv(run / "sample.csv", [
                {"ip_address": "8.8.8.8", "static_country": "United States"},
                {"ip_address": "1.1.1.1", "static_country": "Australia"},
            ])
            save_json(run / "lookup.json", {
                "http_status": 200, "started_at_utc": "start",
                "finished_at_utc": "end", "elapsed_seconds": 0.5,
            })
            save_json(run / "response.body", [
                {"query": "1.1.1.1", "status": "success", "country": "Canada", "countryCode": "CA"},
                {"query": "8.8.8.8", "status": "success", "country": "United States", "countryCode": "US"},
            ])
            # This deliberately differs from the raw response: review must
            # recalculate rather than trusting a previously saved summary.
            save_json(run / "summary.json", {"matches": 99})
            before = {path.name: path.read_bytes() for path in run.iterdir()}
            output = io.StringIO()
            fetch = Mock(side_effect=AssertionError("Review must not make an API call"))
            with patch.dict(globals(), {"fetch_batch": fetch}), redirect_stdout(output):
                summary = review(run)
            fetch.assert_not_called()
            self.assertEqual(summary["matches"], 1)
            self.assertEqual(summary["country_name_match_rate"], 0.5)
            self.assertIn("mismatch", output.getvalue())
            self.assertEqual(before, {path.name: path.read_bytes() for path in run.iterdir()})

In [21]:
sample_test_output = io.StringIO()
sample_test_suite = unittest.defaultTestLoader.loadTestsFromTestCase(SampleTests)
sample_test_result = unittest.TextTestRunner(stream=sample_test_output, verbosity=2).run(sample_test_suite)
print(sample_test_output.getvalue())
assert sample_test_result.wasSuccessful(), "Debug the failing function above, then rerun these notebook checks."

test_collection_preserves_body_and_stops_during_rate_limit_cooldown (__main__.SampleTests.test_collection_preserves_body_and_stops_during_rate_limit_cooldown) ... ok
test_comparison_pairs_by_query_and_excludes_failures_from_denominator (__main__.SampleTests.test_comparison_pairs_by_query_and_excludes_failures_from_denominator) ... ok
test_floor_uses_exact_decimal_and_validates_before_conversion (__main__.SampleTests.test_floor_uses_exact_decimal_and_validates_before_conversion) ... ok
test_public_excludes_special_ranges (__main__.SampleTests.test_public_excludes_special_ranges) ... ok
test_review_recomputes_saved_response_without_network_or_writes (__main__.SampleTests.test_review_recomputes_saved_response_without_network_or_writes) ... ok
test_selection_deduplicates_after_flooring (__main__.SampleTests.test_selection_deduplicates_after_flooring) ... ok
test_static_inclusive_boundaries_and_gaps (__main__.SampleTests.test_static_inclusive_boundaries_and_gaps) ... ok

-------------------

## Verify that inputs and saved evidence stayed unchanged

Compare fingerprints from the start with the current bytes. Also compare the
original CSV hashes with the hashes recorded during collection. This checks
that we reviewed the same source inputs as the successful run. The notebook
itself is the only analysis file being updated.

In [22]:
hashes_after = file_fingerprints(protected_files)
assert hashes_before == hashes_after, "A protected file changed during this walkthrough."
current_evidence_files = sorted(path for path in (RAW_DIR / "ip_api_sample").rglob("*")
                                if path.is_file())
assert current_evidence_files == evidence_files_before, "Saved evidence files changed."

integrity_rows = []
for path in [FRAUD_PATH, STATIC_PATH]:
    current_hash = hashes_after[str(path.relative_to(PROJECT_ROOT))]
    assert current_hash == provenance["source_sha256"][path.name]
    integrity_rows.append({"file": path.name, "unchanged": True, "sha256": current_hash})
display(pd.DataFrame(integrity_rows))
print("Verified: source CSVs, original quick notebook, saved responses, and cooldown record are unchanged.")
print("This walkthrough made no API calls.")

Verified: source CSVs, original quick notebook, saved responses, and cooldown record are unchanged.
This walkthrough made no API calls.


,file,unchanged,sha256
0,Fraud_Data.csv,True,467b1144f3fe03f094816f04aee70434c51389c1ef1999b9d1927bb6bce72c74
1,IpAddress_to_Country.csv,True,720d44848a2b965d17cf4d69e597fb5dbdcef65b78249a29a2f644a9a260594d


## Optional: collect one new 10-IP sample from this notebook

Leave `RUN_LIVE_SAMPLE = False` when reviewing or debugging the completed sample.
With this default, Run All makes no API requests and adds no raw-data files.

If another sample is deliberately needed later, change the flag to `True` and
run this cell once. It checks the cooldown, sends one batch, and saves a new
folder. It does not retry or follow redirects. The folder and summary are printed.
Set `RUN_DIR` above explicitly if you later want to review another saved run,
then restore the flag to `False`.

Full collection is outside this experiment. Both original CSVs must remain
unchanged even when this optional cell is enabled.

In [23]:
RUN_LIVE_SAMPLE = False

if RUN_LIVE_SAMPLE:
    collection_source_hashes = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in [FRAUD_PATH, STATIC_PATH]
    }
    try:
        collection_succeeded = collect(selected, collection_source_hashes)
        print("All 10 country results succeeded:", collection_succeeded)
    finally:
        for path in [FRAUD_PATH, STATIC_PATH]:
            assert hashlib.sha256(path.read_bytes()).hexdigest() == collection_source_hashes[path.name]
else:
    print("Live collection is OFF. We are reviewing the saved sample only.")

Live collection is OFF. We are reviewing the saved sample only.


## Updated findings and remaining questions

- The successful batch at **2026-10-31 19:45:57 UTC** returned **10 countries**,
  with **zero failed or incomplete results**. Eight match and two differ:
  **80% agreement on this convenience sample**.
- `43.173.1.96`: static **Japan**, API **Indonesia**. `20.93.138.243`: static
  **United States**, API **Netherlands**. Keep both sources; their historical
  correctness remains unconfirmed.
- The batch took **0.240207 seconds**. The original response is **1,175 bytes**;
  headers were `X-Rl: 14` and `X-Ttl: 60`. Reviewing and debugging the saved result
  needs no new request.
- Original IP text is preserved. Flooring changes zero static assignments in
  this sample and the full-dataset diagnostic, but the intended encoding still
  needs confirmation. API success does not prove the assumption.
- The DNS failure and ISP redirect remain collection-quality evidence. Their
  agreement rates are unavailable, not zero.
- All seven offline checks are now here. Original CSVs and saved responses stay
  unchanged.

**Data understanding remains open.** Review the country differences and encoding,
investigate the one-second fraud pattern, and resolve the recorded questions
about labels, currency, units, and timezones. Full collection and CPI scraping
are still deferred. Until this phase is finished, keep all analysis, code,
experiments, and debugging in this notebook.